# IronClad Task 3: Indexing Selection for `vggface2` with Minkowski metric

This notebook supplements the main indexing analysis (which uses cosine similarity as the native FAISS metric) by repeating the same architecture comparison under the Minkowski distance with $p = 3$. 

Because Minkowski is not available as a native FAISS metric, the retrieval pipeline differs from the cosine case: each index first retrieves a candidate set using its own internal search mechanism, and those candidates are then reranked by their exact Minkowski distance to the probe. 

The notebook therefore evaluates which indexing strategy (Brute Force, HNSW, or LSH) best preserves the exact Minkowski neighbor ranking after reranking. The analysis follows the same structure as the cosine-based notebook: hyperparameter tuning on the real IronClad workload, a full real-task benchmark, real-data scaling with identity-preserving subsampling, a synthetic large-scale benchmark, and extrapolation to $10^9$ vectors. 

The purpose is to determine whether Minkowski remains a viable deployment metric at scale, or whether the overhead of the retrieval-plus-reranking pipeline makes it impractical relative to the cosine-based alternative.

In [1]:

import sys, subprocess, importlib, os, json, gc, time, math
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

def ensure_pkg(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        return importlib.import_module(import_name)

faiss = ensure_pkg("faiss")
ensure_pkg("facenet_pytorch")
ensure_pkg("torch")
ensure_pkg("PIL")

from PIL import Image, UnidentifiedImageError
from facenet_pytorch import MTCNN


In [2]:

PROJECT_ROOT = Path.cwd().parent  # assume notebook runs from BASE/analysis
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ironclad.modules.extraction.preprocessing import Preprocessing
from ironclad.modules.extraction.embedding import Embedding
from ironclad.modules.retrieval.index.bruteforce import FaissBruteForce
from ironclad.modules.retrieval.index.hnsw import FaissHNSW
from ironclad.modules.retrieval.index.lsh import FaissLSH

print("Project root:", PROJECT_ROOT)
DEVICE = "cuda" if importlib.import_module("torch").cuda.is_available() else "cpu"
print("Device:", DEVICE)

BEST_MODEL = "vggface2"
PREPROCESS_VARIANT = "mtcnn"
MTCNN_THRESHOLD = 0.80
MTCNN_MARGIN = 20
MINKOWSKI_P = 3
CANDIDATE_MULT = 25        # used to request enough candidates to rerank identities
MIN_CANDIDATES = 100


Project root: C:\Users\Usuario\Downloads\ironclad-hortner87-main(1)\ironclad-hortner87-main
Device: cpu


In [3]:

STORAGE_ROOT = PROJECT_ROOT / "ironclad" / "storage"
GALLERY_DIR = STORAGE_ROOT / "gallery"
PROBE_DIR = STORAGE_ROOT / "probe"

print("Storage root:", STORAGE_ROOT)
print("Gallery dir :", GALLERY_DIR)
print("Probe dir   :", PROBE_DIR)

def list_images_by_identity(base_dir: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    out = {}
    for ident_dir in sorted([p for p in base_dir.iterdir() if p.is_dir()]):
        imgs = []
        for p in ident_dir.rglob("*"):
            if not p.is_file():
                continue
            if p.suffix.lower() not in exts:
                continue
            if p.name.startswith("._") or p.name.startswith("."):
                continue
            imgs.append(p)
        if imgs:
            out[ident_dir.name] = sorted(imgs)
    return out

gallery = list_images_by_identity(GALLERY_DIR)
probe = list_images_by_identity(PROBE_DIR)
overlap = sorted(set(gallery.keys()) & set(probe.keys()))

print("Gallery identities:", len(gallery))
print("Probe identities  :", len(probe))
print("Overlap identities:", len(overlap))
print("Example overlap   :", overlap[:5])


Storage root: C:\Users\Usuario\Downloads\ironclad-hortner87-main(1)\ironclad-hortner87-main\ironclad\storage
Gallery dir : C:\Users\Usuario\Downloads\ironclad-hortner87-main(1)\ironclad-hortner87-main\ironclad\storage\gallery
Probe dir   : C:\Users\Usuario\Downloads\ironclad-hortner87-main(1)\ironclad-hortner87-main\ironclad\storage\probe
Gallery identities: 1000
Probe identities  : 999
Overlap identities: 999
Example overlap   : ['Aaron_Sorkin', 'Abdel_Nasser_Assidi', 'Abdullah', 'Abdullah_Gul', 'Abdullah_al-Attiyah']


In [4]:

preproc = Preprocessing(image_size=160)
CACHE_DIR = Path.cwd() / "cache_mtcnn_shared"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

mtcnn = MTCNN(
    image_size=160,
    margin=MTCNN_MARGIN,
    keep_all=True,
    device=DEVICE
)

def pil_load_rgb(path: Path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError):
        return None

DETECTION_CACHE_PATH = CACHE_DIR / "mtcnn_detection_cache.json"
if DETECTION_CACHE_PATH.exists():
    with open(DETECTION_CACHE_PATH, "r", encoding="utf-8") as f:
        DET_CACHE = json.load(f)
else:
    DET_CACHE = {}

def _save_det_cache():
    with open(DETECTION_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(DET_CACHE, f)

def detect_best_face(path: Path):
    key = str(path)
    if key in DET_CACHE:
        rec = DET_CACHE[key]
        box = rec["box"]
        prob = rec["prob"]
        return (np.array(box, dtype=float) if box is not None else None), prob

    img = pil_load_rgb(path)
    if img is None:
        DET_CACHE[key] = {"box": None, "prob": None}
        _save_det_cache()
        return None, None

    boxes, probs = mtcnn.detect(img)
    if boxes is None or probs is None or len(boxes) == 0:
        DET_CACHE[key] = {"box": None, "prob": None}
        _save_det_cache()
        return None, None

    best = int(np.argmax(probs))
    box = boxes[best].tolist()
    prob = float(probs[best]) if probs[best] is not None else None
    DET_CACHE[key] = {"box": box, "prob": prob}
    _save_det_cache()
    return np.array(box, dtype=float), prob

def crop_from_box(img: Image.Image, box, margin_px=MTCNN_MARGIN):
    x1, y1, x2, y2 = [float(v) for v in box]
    x1 = max(0, int(np.floor(x1)) - margin_px)
    y1 = max(0, int(np.floor(y1)) - margin_px)
    x2 = min(img.width, int(np.ceil(x2)) + margin_px)
    y2 = min(img.height, int(np.ceil(y2)) + margin_px)
    if x2 <= x1 or y2 <= y1:
        return img
    return img.crop((x1, y1, x2, y2))

def get_mtcnn_variant_image(path: Path, threshold: float = MTCNN_THRESHOLD):
    img = pil_load_rgb(path)
    if img is None:
        return None, None, False
    box, prob = detect_best_face(path)
    use_crop = box is not None and prob is not None and prob >= threshold
    if use_crop:
        return crop_from_box(img, box, margin_px=MTCNN_MARGIN), prob, True
    return img, prob, False

def l2_normalize(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)

def embed_split_mtcnn(model_name: str, paths, labels, split_name: str, threshold: float = MTCNN_THRESHOLD):
    key_src = f"{model_name}|{split_name}|mtcnn|{threshold}|{len(paths)}"
    key = str(abs(hash(key_src)))
    cache_path = CACHE_DIR / f"emb_{model_name}_{split_name}_mtcnn_{key}.npz"
    if cache_path.exists():
        z = np.load(cache_path, allow_pickle=True)
        return (
            z["X"].astype(np.float32),
            z["labels"].tolist(),
            [Path(p) for p in z["paths"].tolist()],
            dict(
                crop_rate=float(z["crop_rate"][0]),
                mean_conf=float(z["mean_conf"][0]) if len(z["mean_conf"]) else np.nan,
                n_images=int(z["n_images"][0]),
            )
        )

    model = Embedding(pretrained=model_name, device=DEVICE)
    vecs, labs, keep_paths = [], [], []
    confs, used_crop = [], []
    for p, lab in tqdm(list(zip(paths, labels)), desc=f"Embedding {split_name} | {model_name} | mtcnn"):
        img_variant, conf, did_crop = get_mtcnn_variant_image(p, threshold=threshold)
        if img_variant is None:
            continue
        tens = preproc.process(img_variant)
        vec = model.encode(tens).astype(np.float32)
        vecs.append(vec)
        labs.append(lab)
        keep_paths.append(str(p))
        if conf is not None:
            confs.append(float(conf))
        used_crop.append(bool(did_crop))

    X = np.vstack([v.reshape(1, -1) for v in vecs]).astype(np.float32)
    X = l2_normalize(X)   # keep normalized embeddings, as in the cosine-based notebook
    crop_rate = float(np.mean(used_crop)) if used_crop else 0.0
    mean_conf = float(np.mean(confs)) if confs else np.nan

    np.savez(
        cache_path,
        X=X,
        labels=np.array(labs, dtype=object),
        paths=np.array(keep_paths, dtype=object),
        crop_rate=np.array([crop_rate], dtype=np.float32),
        mean_conf=np.array([mean_conf], dtype=np.float32),
        n_images=np.array([len(labs)], dtype=np.int32),
    )
    return X, labs, [Path(p) for p in keep_paths], dict(crop_rate=crop_rate, mean_conf=mean_conf, n_images=len(labs))



## Prepare embeddings

This notebook fixes the system to the best upstream design identified earlier: `vggface2 + MTCNN`. Only the retrieval **metric/indexing backend** is changed.


In [5]:

gallery_paths, gallery_labels = [], []
for ident in overlap:
    for p in gallery[ident]:
        gallery_paths.append(p)
        gallery_labels.append(ident)

probe_paths, probe_labels = [], []
for ident in overlap:
    for p in probe[ident]:
        probe_paths.append(p)
        probe_labels.append(ident)

gal_X, gal_y, gal_keep_paths, gal_meta = embed_split_mtcnn(
    BEST_MODEL, gallery_paths, gallery_labels, split_name="gallery_clean"
)
prb_X, prb_y, prb_keep_paths, prb_meta = embed_split_mtcnn(
    BEST_MODEL, probe_paths, probe_labels, split_name="probe_clean"
)

print("Gallery embeddings:", gal_X.shape)
print("Probe embeddings  :", prb_X.shape)
print("Gallery crop rate :", gal_meta["crop_rate"], "| mean conf:", gal_meta["mean_conf"])
print("Probe crop rate   :", prb_meta["crop_rate"], "| mean conf:", prb_meta["mean_conf"])


Embedding gallery_clean | vggface2 | mtcnn:   0%|          | 0/2261 [00:00<?, ?it/s]

Embedding probe_clean | vggface2 | mtcnn:   0%|          | 0/999 [00:00<?, ?it/s]

Gallery embeddings: (2261, 512)
Probe embeddings  : (999, 512)
Gallery crop rate : 0.9995577178239717 | mean conf: 0.9996893697871571
Probe crop rate   : 1.0 | mean conf: 0.9995434910088807


In [6]:

def minkowski_distance_matrix(Q: np.ndarray, X: np.ndarray, p: int = 3) -> np.ndarray:
    # Q: (nq, d), X: (ng, d)
    # returns distances shape (nq, ng)
    return np.sum(np.abs(Q[:, None, :] - X[None, :, :]) ** p, axis=2) ** (1.0 / p)

def exact_minkowski_knn(Q: np.ndarray, X: np.ndarray, k: int, p: int = 3) -> np.ndarray:
    D = minkowski_distance_matrix(Q, X, p=p)
    I = np.argpartition(D, kth=min(k-1, D.shape[1]-1), axis=1)[:, :k]
    # sort the selected candidates
    row = np.arange(D.shape[0])[:, None]
    order = np.argsort(D[row, I], axis=1)
    return I[row, order]

def build_candidate_indices(gal_X, gal_y, hnsw_efSearch=16, lsh_nbits=512):
    # Candidate generators; BF/HNSW use euclidean source for consistency with project Minkowski logic
    bf = FaissBruteForce(dim=gal_X.shape[1], metric="euclidean")
    t0 = time.perf_counter(); bf.add_embeddings(gal_X, gal_y); t1 = time.perf_counter()
    t_bf = t1 - t0

    hnsw = FaissHNSW(dim=gal_X.shape[1], metric="euclidean", M=32, efConstruction=40)
    t0 = time.perf_counter(); hnsw.add_embeddings(gal_X, gal_y); t1 = time.perf_counter()
    hnsw.index.hnsw.efSearch = hnsw_efSearch
    t_h = t1 - t0

    lsh = FaissLSH(dim=gal_X.shape[1], nbits=lsh_nbits)
    t0 = time.perf_counter(); lsh.add_embeddings(gal_X, gal_y); t1 = time.perf_counter()
    t_l = t1 - t0
    return (bf, hnsw, lsh), (t_bf, t_h, t_l)

def rerank_candidates_minkowski(query_vec: np.ndarray, cand_idx: np.ndarray, gal_X: np.ndarray, p: int = 3):
    cand_idx = np.asarray(cand_idx, dtype=int)
    cand_idx = cand_idx[cand_idx >= 0]
    if cand_idx.size == 0:
        return cand_idx
    Xc = gal_X[cand_idx]
    d = np.sum(np.abs(Xc - query_vec[None, :]) ** p, axis=1) ** (1.0 / p)
    return cand_idx[np.argsort(d)]

def batch_search_minkowski(index_name: str, index_obj, Q: np.ndarray, gal_X: np.ndarray, topk: int = 10,
                           candidate_k: int | None = None, p: int = 3):
    n_gallery = gal_X.shape[0]
    if index_name == "bruteforce":
        I = exact_minkowski_knn(Q, gal_X, k=topk, p=p)
        return I

    candidate_k = candidate_k or min(n_gallery, max(MIN_CANDIDATES, topk * CANDIDATE_MULT))
    candidate_k = min(candidate_k, n_gallery)

    _, I_cand = index_obj.index.search(Q.astype(np.float32), candidate_k)
    out = np.full((Q.shape[0], topk), -1, dtype=int)
    for qi in range(Q.shape[0]):
        reranked = rerank_candidates_minkowski(Q[qi], I_cand[qi], gal_X, p=p)
        use = reranked[:topk]
        out[qi, :len(use)] = use
    return out

def topk_acc_from_indices(I, metadata, true_labels, k=1):
    correct = 0
    for qi in range(I.shape[0]):
        preds = [metadata[int(j)] for j in I[qi, :k] if int(j) >= 0]
        if true_labels[qi] in preds:
            correct += 1
    return correct / len(true_labels)

def recall_at_k_vs_exact(approx_i, exact_i, k=10):
    hits = 0
    nq = exact_i.shape[0]
    for qi in range(nq):
        if int(exact_i[qi, 0]) in set(map(int, approx_i[qi, :k])):
            hits += 1
    return hits / nq



## Tune HNSW and LSH on a subset of the IronClad dataset

As in the cosine notebook, ANN settings are tuned on the real task before the final comparison. The quality reference is now the **exact Minkowski neighbor set** rather than the cosine/FAISS exact search.


In [7]:

def tune_hnsw_and_lsh_minkowski(gal_X, gal_y, prb_X, prb_y, ef_candidates=(16, 32, 64, 128), nbits_candidates=(128, 256, 512), p=3):
    rows = []
    I_ref = exact_minkowski_knn(prb_X, gal_X, k=10, p=p)

    for ef in ef_candidates:
        h = FaissHNSW(dim=gal_X.shape[1], metric="euclidean", M=32, efConstruction=40)
        h.add_embeddings(gal_X, gal_y)
        h.index.hnsw.efSearch = ef
        t0 = time.perf_counter()
        Ih = batch_search_minkowski("hnsw", h, prb_X, gal_X, topk=10, p=p)
        t1 = time.perf_counter()
        rows.append({
            "index": "hnsw",
            "setting": ef,
            "ms_per_query": (t1 - t0) * 1000.0 / prb_X.shape[0],
            "top1": topk_acc_from_indices(Ih, h.metadata, prb_y, 1),
            "top5": topk_acc_from_indices(Ih, h.metadata, prb_y, 5),
            "recall10_vs_exact_minkowski": recall_at_k_vs_exact(Ih, I_ref, 10),
        })

    for nb in nbits_candidates:
        l = FaissLSH(dim=gal_X.shape[1], nbits=nb)
        l.add_embeddings(gal_X, gal_y)
        t0 = time.perf_counter()
        Il = batch_search_minkowski("lsh", l, prb_X, gal_X, topk=10, p=p)
        t1 = time.perf_counter()
        rows.append({
            "index": "lsh",
            "setting": nb,
            "ms_per_query": (t1 - t0) * 1000.0 / prb_X.shape[0],
            "top1": topk_acc_from_indices(Il, l.metadata, prb_y, 1),
            "top5": topk_acc_from_indices(Il, l.metadata, prb_y, 5),
            "recall10_vs_exact_minkowski": recall_at_k_vs_exact(Il, I_ref, 10),
        })

    return pd.DataFrame(rows)

tune_df = tune_hnsw_and_lsh_minkowski(gal_X, gal_y, prb_X, prb_y, p=MINKOWSKI_P)
tune_df


,index,setting,ms_per_query,top1,top5,recall10_vs_exact_minkowski
0,hnsw,16,1.432339,0.842843,0.899900,0.994995
1,hnsw,32,1.418696,0.844845,0.902903,1.000000
2,hnsw,64,1.506958,0.844845,0.902903,1.000000
3,hnsw,128,1.692091,0.844845,0.902903,1.000000
4,lsh,128,1.427195,0.844845,0.901902,0.997998
5,lsh,256,1.392457,0.844845,0.902903,1.000000
6,lsh,512,1.383985,0.844845,0.902903,1.000000


In [8]:

def choose_setting(df_sub, index_name):
    sub = df_sub[df_sub["index"] == index_name].sort_values(
        ["top1", "top5", "recall10_vs_exact_minkowski", "ms_per_query"],
        ascending=[False, False, False, True]
    )
    return int(sub.iloc[0]["setting"])

chosen_hnsw_efSearch = choose_setting(tune_df, "hnsw")
chosen_lsh_nbits = choose_setting(tune_df, "lsh")

chosen_df = pd.DataFrame([{
    "model": BEST_MODEL,
    "preprocessing": PREPROCESS_VARIANT,
    "metric": f"minkowski_p{MINKOWSKI_P}",
    "hnsw_efSearch": chosen_hnsw_efSearch,
    "lsh_nbits": chosen_lsh_nbits
}])
chosen_df


,model,preprocessing,metric,hnsw_efSearch,lsh_nbits
0,vggface2,mtcnn,minkowski_p3,32,512



## Full IronClad dataset benchmark

This section matches the Task 3 notebook for cosine metric:
- full gallery
- full probe set
- Top-1 / Top-5
- recall against the exact Minkowski ranking
- build time
- query latency


In [9]:

(bf, hnsw, lsh), (t_bf, t_h, t_l) = build_candidate_indices(
    gal_X, gal_y, hnsw_efSearch=chosen_hnsw_efSearch, lsh_nbits=chosen_lsh_nbits
)

# Exact Minkowski reference
t0 = time.perf_counter()
I_bf = batch_search_minkowski("bruteforce", bf, prb_X, gal_X, topk=10, p=MINKOWSKI_P)
t1 = time.perf_counter()
ms_bf = (t1 - t0) * 1000.0 / prb_X.shape[0]

t0 = time.perf_counter()
I_h = batch_search_minkowski("hnsw", hnsw, prb_X, gal_X, topk=10, p=MINKOWSKI_P)
t1 = time.perf_counter()
ms_h = (t1 - t0) * 1000.0 / prb_X.shape[0]

t0 = time.perf_counter()
I_l = batch_search_minkowski("lsh", lsh, prb_X, gal_X, topk=10, p=MINKOWSKI_P)
t1 = time.perf_counter()
ms_l = (t1 - t0) * 1000.0 / prb_X.shape[0]

real_bench_df = pd.DataFrame([
    dict(index="bruteforce", build_s=t_bf, ms_per_query=ms_bf,
         top1=topk_acc_from_indices(I_bf, bf.metadata, prb_y, 1),
         top5=topk_acc_from_indices(I_bf, bf.metadata, prb_y, 5),
         recall10_vs_exact_minkowski=1.0, n_gallery=gal_X.shape[0], n_probe=prb_X.shape[0]),
    dict(index="hnsw", build_s=t_h, ms_per_query=ms_h,
         top1=topk_acc_from_indices(I_h, hnsw.metadata, prb_y, 1),
         top5=topk_acc_from_indices(I_h, hnsw.metadata, prb_y, 5),
         recall10_vs_exact_minkowski=recall_at_k_vs_exact(I_h, I_bf, 10), n_gallery=gal_X.shape[0], n_probe=prb_X.shape[0]),
    dict(index="lsh", build_s=t_l, ms_per_query=ms_l,
         top1=topk_acc_from_indices(I_l, lsh.metadata, prb_y, 1),
         top5=topk_acc_from_indices(I_l, lsh.metadata, prb_y, 5),
         recall10_vs_exact_minkowski=recall_at_k_vs_exact(I_l, I_bf, 10), n_gallery=gal_X.shape[0], n_probe=prb_X.shape[0]),
])
real_bench_df


,index,build_s,ms_per_query,top1,top5,recall10_vs_exact_minkowski,n_gallery,n_probe
0,bruteforce,0.001861,16.116062,0.844845,0.902903,1.0,2261,999
1,hnsw,0.039277,1.718604,0.844845,0.902903,1.0,2261,999
2,lsh,0.007285,1.438859,0.844845,0.902903,1.0,2261,999



## Scaling on real data

We keep the same scaling design as the cosine notebook: every identity remains represented by at least one anchor gallery image, and extra gallery images are added as the target gallery size grows.


In [10]:

def identity_to_gallery_indices(labels):
    groups = {}
    for i, ident in enumerate(labels):
        groups.setdefault(ident, []).append(i)
    return groups

def make_scaled_subset_indices(labels, target_total, seed=42):
    groups = identity_to_gallery_indices(labels)
    rng = np.random.default_rng(seed)
    anchors = [idxs[0] for idxs in groups.values()]
    if target_total <= len(anchors):
        return sorted(anchors[:target_total])
    extra_pool = []
    for idxs in groups.values():
        extra_pool.extend(idxs[1:])
    extra_needed = min(target_total - len(anchors), len(extra_pool))
    extras = rng.choice(np.array(extra_pool), size=extra_needed, replace=False).tolist() if extra_needed > 0 else []
    return sorted(anchors + extras)

def benchmark_scaled_minkowski(gal_X_full, gal_y_full, prb_X, prb_y, targets, hnsw_efSearch, lsh_nbits, p=3):
    rows = []
    for N in targets:
        sel = make_scaled_subset_indices(gal_y_full, target_total=N, seed=42)
        Xg = gal_X_full[sel]
        yg = [gal_y_full[i] for i in sel]
        (bf_s, h_s, l_s), _ = build_candidate_indices(Xg, yg, hnsw_efSearch=hnsw_efSearch, lsh_nbits=lsh_nbits)

        t0 = time.perf_counter()
        I_bf_s = batch_search_minkowski("bruteforce", bf_s, prb_X, Xg, topk=10, p=p)
        t1 = time.perf_counter()
        ms_bf_s = (t1 - t0) * 1000.0 / prb_X.shape[0]

        t0 = time.perf_counter()
        I_h_s = batch_search_minkowski("hnsw", h_s, prb_X, Xg, topk=10, p=p)
        t1 = time.perf_counter()
        ms_h_s = (t1 - t0) * 1000.0 / prb_X.shape[0]

        t0 = time.perf_counter()
        I_l_s = batch_search_minkowski("lsh", l_s, prb_X, Xg, topk=10, p=p)
        t1 = time.perf_counter()
        ms_l_s = (t1 - t0) * 1000.0 / prb_X.shape[0]

        rows.extend([
            dict(N=N, index="bruteforce", ms_per_query=ms_bf_s,
                 top1=topk_acc_from_indices(I_bf_s, bf_s.metadata, prb_y, 1),
                 top5=topk_acc_from_indices(I_bf_s, bf_s.metadata, prb_y, 5),
                 recall10_vs_exact_minkowski=1.0),
            dict(N=N, index="hnsw", ms_per_query=ms_h_s,
                 top1=topk_acc_from_indices(I_h_s, h_s.metadata, prb_y, 1),
                 top5=topk_acc_from_indices(I_h_s, h_s.metadata, prb_y, 5),
                 recall10_vs_exact_minkowski=recall_at_k_vs_exact(I_h_s, I_bf_s, 10)),
            dict(N=N, index="lsh", ms_per_query=ms_l_s,
                 top1=topk_acc_from_indices(I_l_s, l_s.metadata, prb_y, 1),
                 top5=topk_acc_from_indices(I_l_s, l_s.metadata, prb_y, 5),
                 recall10_vs_exact_minkowski=recall_at_k_vs_exact(I_l_s, I_bf_s, 10)),
        ])
    return pd.DataFrame(rows)

fullN = gal_X.shape[0]
targets = sorted(set([len(overlap), int(fullN*0.25), int(fullN*0.50), int(fullN*0.75), fullN]))
targets = [t for t in targets if t >= len(overlap)]

scaled_df = benchmark_scaled_minkowski(
    gal_X, gal_y, prb_X, prb_y, targets,
    hnsw_efSearch=chosen_hnsw_efSearch,
    lsh_nbits=chosen_lsh_nbits,
    p=MINKOWSKI_P
)
scaled_df


,N,index,ms_per_query,top1,top5,recall10_vs_exact_minkowski
0,999,bruteforce,6.618323,0.819820,0.892893,1.000000
1,999,hnsw,1.402928,0.819820,0.892893,1.000000
2,999,lsh,1.506520,0.819820,0.892893,1.000000
3,1130,bruteforce,7.259750,0.828829,0.899900,1.000000
4,1130,hnsw,1.374554,0.828829,0.899900,0.998999
5,1130,lsh,1.645516,0.828829,0.899900,1.000000
6,1695,bruteforce,12.702405,0.836837,0.902903,1.000000
7,1695,hnsw,1.401240,0.836837,0.902903,1.000000
8,1695,lsh,1.374044,0.836837,0.902903,1.000000
9,2261,bruteforce,15.377294,0.844845,0.902903,1.000000



## Synthetic large-scale benchmark for the 1B target

This section follows the same logic as the cosine-based notebook: we benchmark the candidate generators at larger synthetic sizes and use those measurements to extrapolate memory and latency toward a 1B-vector gallery.

The key difference is that approximate quality is measured against the **exact Minkowski** neighbor set computed on the same synthetic data.


In [11]:

def serialized_size_bytes(faiss_index) -> int:
    return len(faiss.serialize_index(faiss_index))

def fit_power_law(N, y):
    x = np.log(np.asarray(N, dtype=float))
    yy = np.log(np.asarray(y, dtype=float))
    b, loga = np.polyfit(x, yy, 1)
    return float(np.exp(loga)), float(b)

def run_synthetic_benchmark_minkowski(Ns=(25_000, 75_000, 125_000), d=512, nq=50, k=10,
                                      hnsw_efSearch=16, lsh_nbits=512, p=3):
    rows = []
    rng = np.random.default_rng(123)
    for N in Ns:
        print(f"\n=== Synthetic benchmark N={N:,} ===")
        X = rng.random((N, d), dtype=np.float32)
        Q = rng.random((nq, d), dtype=np.float32)
        X = l2_normalize(X)
        Q = l2_normalize(Q)

        (bf_cand, hnsw_cand, lsh_cand), (t_bf, t_h, t_l) = build_candidate_indices(
            X, [str(i) for i in range(N)], hnsw_efSearch=hnsw_efSearch, lsh_nbits=lsh_nbits
        )

        t0 = time.perf_counter()
        I_bf = batch_search_minkowski("bruteforce", bf_cand, Q, X, topk=k, p=p)
        t1 = time.perf_counter()
        ms_bf = (t1 - t0) * 1000.0 / nq

        t0 = time.perf_counter()
        I_h = batch_search_minkowski("hnsw", hnsw_cand, Q, X, topk=k, p=p)
        t1 = time.perf_counter()
        ms_h = (t1 - t0) * 1000.0 / nq

        t0 = time.perf_counter()
        I_l = batch_search_minkowski("lsh", lsh_cand, Q, X, topk=k, p=p)
        t1 = time.perf_counter()
        ms_l = (t1 - t0) * 1000.0 / nq

        rows.extend([
            dict(index="bruteforce", N=N, build_s=t_bf, ms_per_query=ms_bf,
                 recall_at_k=1.0, serialized_bytes=serialized_size_bytes(bf_cand.index)),
            dict(index="hnsw", N=N, build_s=t_h, ms_per_query=ms_h,
                 recall_at_k=recall_at_k_vs_exact(I_h, I_bf, k), serialized_bytes=serialized_size_bytes(hnsw_cand.index)),
            dict(index="lsh", N=N, build_s=t_l, ms_per_query=ms_l,
                 recall_at_k=recall_at_k_vs_exact(I_l, I_bf, k), serialized_bytes=serialized_size_bytes(lsh_cand.index)),
        ])

        del X, Q, bf_cand, hnsw_cand, lsh_cand, I_bf, I_h, I_l
        gc.collect()

    df = pd.DataFrame(rows)
    df["bytes_per_vec"] = df["serialized_bytes"] / df["N"]
    return df

synth_df = run_synthetic_benchmark_minkowski(
    hnsw_efSearch=chosen_hnsw_efSearch,
    lsh_nbits=chosen_lsh_nbits,
    p=MINKOWSKI_P
)
synth_df



=== Synthetic benchmark N=25,000 ===

=== Synthetic benchmark N=75,000 ===

=== Synthetic benchmark N=125,000 ===


,index,N,build_s,ms_per_query,recall_at_k,serialized_bytes,bytes_per_vec
0,bruteforce,25000,0.010245,158.832042,1.00,51200045,2048.001800
1,hnsw,25000,10.310132,2.444770,0.38,58002882,2320.115280
2,lsh,25000,0.070369,2.247800,0.30,2648669,105.946760
3,bruteforce,75000,0.053202,2792.434120,1.00,153600045,2048.000600
4,hnsw,75000,48.308226,16.888610,0.32,174013186,2320.175813
5,lsh,75000,0.214475,4.517468,0.22,5848669,77.982253
6,bruteforce,125000,0.070888,8559.637108,1.00,256000045,2048.000360
7,hnsw,125000,89.267729,85.534658,0.24,290026946,2320.215568
8,lsh,125000,0.293293,11.814364,0.08,9048669,72.389352


In [12]:

TARGET_N = 1_000_000_000
rows = []
for index_name in synth_df["index"].unique():
    sub = synth_df[synth_df["index"] == index_name].sort_values("N")
    bpp = float(sub.iloc[-1]["bytes_per_vec"])
    mem_gb = bpp * TARGET_N / (1024**3)
    a, b = fit_power_law(sub["N"], sub["ms_per_query"])
    ms_1b = a * (TARGET_N ** b)
    rows.append({
        "index": index_name,
        "bytes_per_vec_est": bpp,
        "mem_for_1B_GB_est": mem_gb,
        "latency_fit_b": b,
        "ms_per_query_at_1B_est": ms_1b,
        "recall_at_largest_N": float(sub.iloc[-1]["recall_at_k"]),
    })
scale_df = pd.DataFrame(rows).sort_values("ms_per_query_at_1B_est")
scale_df


,index,bytes_per_vec_est,mem_for_1B_GB_est,latency_fit_b,ms_per_query_at_1B_est,recall_at_largest_N
2,lsh,72.389352,67.417838,0.968060,5.869250e+04,0.08
1,hnsw,2320.215568,2160.869136,2.137283,1.515939e+10,0.24
0,bruteforce,2048.000360,1907.348968,2.498285,5.140215e+13,1.00
